In [ ]:
# 1. Setup paths and mount Drive
from google.colab import drive
from pathlib import Path
import os

# Mount Drive
drive.mount('/content/drive')

# Path definitions - Drive (persistent)
WORKSPACE = '/content/drive/MyDrive/evo1_baseline'
RESULTS_DIR = Path(f'{WORKSPACE}/Results')

# Path definitions - Local (ephemeral)
CHECKPOINTS_DIR = Path('/content/checkpoints')
LOGS_DIR = Path('/content/logs')

# Create directories
for d in [RESULTS_DIR, CHECKPOINTS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Clear PYTHONPATH
os.environ.pop('PYTHONPATH', None)
os.environ['MUJOCO_GL'] = 'egl'
os.environ.pop('DISPLAY', None)

# Check GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError("❌ No GPU! Enable in Runtime > Change runtime type")

gpu_name = torch.cuda.get_device_name(0)
total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"✅ GPU: {gpu_name} ({total_mem:.1f} GB)")
print(f"✅ Workspace: {WORKSPACE}")
print(f"✅ Results: {RESULTS_DIR}")

In [ ]:
# ISSUE DISCOVERED: 2026-01-28
# - Transformers 5.0.0 released on 2026-01-26
# - New version changes model loading behavior (more aggressive meta device usage)
# - Causes "RuntimeError: Tensor.item() cannot be called on meta tensors"
# - Error occurs inside InternVL3 model's own initialization code
# 
# ROOT CAUSE:
# - InternVL3 model code uses torch.linspace() during __init__
# - Transformers 5.0 creates tensors on meta device by default for memory efficiency
# - Meta tensors can't call .item() method, causing initialization to fail
# 
# SOLUTION:
# - Downgrade to transformers 4.46.3 (last stable 4.x release)
# - Clear HuggingFace cache to remove any cached model code using 5.0 APIs
# 
# REFERENCE:
# - Working version: transformers 4.XX.X
# - Broken version: transformers 5.0.0+
# - Model: OpenGVLab/InternVL3-1B (used by Evo-1)

In [ ]:
# 2. Install dependencies (3 SEPARATE conda environments - official structure)
print('📦 Installing system dependencies...')
!apt-get update -qq
!apt-get install -y -qq git wget ffmpeg libosmesa6-dev patchelf libgl1-mesa-glx libegl1-mesa > /dev/null 2>&1

print('📦 Installing Miniconda...')
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
!bash /tmp/miniconda.sh -b -p /opt/conda > /dev/null 2>&1
!rm /tmp/miniconda.sh
os.environ['PATH'] = f"/opt/conda/bin:{os.environ['PATH']}"
!conda init bash
!conda config --set always_yes yes

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

print('\n📦 Creating 3 conda environments (official Evo-1 structure)...')
print('='*60)

# Environment 1: Evo-1 server (Python 3.10)
print('\n[1/3] evo1_server (Python 3.10) - for Evo-1 model')
!conda create -n evo1_server python=3.10 -y
!conda run -n evo1_server pip install  \
  torch==2.5.1+cu121 torchvision==0.20.1+cu121 \
  --index-url https://download.pytorch.org/whl/cu121
!conda run -n evo1_server pip install \
  'numpy>=1.26.4,<2.0' 'transformers == 4.57.6' accelerate diffusers \
  einops timm pillow opencv-python-headless \
  websockets pyyaml huggingface_hub tqdm fvcore
print('📦 Installing flash-attn (required, ~10-15 min)...')
!conda run -n evo1_server pip install flash-attn --no-build-isolation 2>&1 | tail -5 || echo '⚠️ Flash-attn failed'
print('✅ evo1_server ready')

# Environment 2: LIBERO client (Python 3.8.13 - OFFICIAL requirement)
print('\n[2/3] libero_client (Python 3.8.13) - for LIBERO benchmark')
!conda create -n libero_client python=3.8.13 -y
!conda run -n libero_client pip install \
  'numpy>=1.20,<2.0' robosuite==1.4.1 mujoco==2.3.7 \
  imageio h5py bddl websockets huggingface_hub
!conda run -n libero_client pip install \
  torch==1.11.0+cu113 torchvision==0.12.0+cu113 torchaudio==0.11.0 \
  --extra-index-url https://download.pytorch.org/whl/cu113
print('✅ libero_client ready')

# Environment 3: MetaWorld client (Python 3.10)
print('\n[3/3] metaworld_client (Python 3.10) - for MetaWorld benchmark')
!conda create -n metaworld_client python=3.10 -y
!conda run -n metaworld_client pip install \
  mujoco websockets opencv-python packaging huggingface_hub metaworld gymnasium

print('✅ metaworld_client ready')

print('\n' + '='*60)
print('✅ All 3 environments created!')
!conda run -n evo1_server python -c "import sys; print(f'  evo1_server: Python {sys.version.split()[0]}')"
!conda run -n libero_client python -c "import sys; print(f'  libero_client: Python {sys.version.split()[0]}')"
!conda run -n metaworld_client python -c "import sys; print(f'  metaworld_client: Python {sys.version.split()[0]}')"

📦 Installing system dependencies...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
📦 Installing Miniconda...
no change     /opt/conda/condabin/conda
no change     /opt/conda/bin/conda
no change     /opt/conda/bin/conda-env
no change     /opt/conda/bin/activate
no change     /opt/conda/bin/deactivate
no change     /opt/conda/etc/profile.d/conda.sh
no change     /opt/conda/etc/fish/conf.d/conda.fish
no change     /opt/conda/shell/condabin/Conda.psm1
no change     /opt/conda/shell/condabin/conda-hook.ps1
no change     /opt/conda/lib/python3.13/site-packages/xontrib/conda.xsh
no change     /opt/conda/etc/profile.d/conda.csh
no change     /root/.bashrc
No action taken.
accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r

📦 Creating 3 conda environments (official Evo-1 structu

In [ ]:
# 3. Clone and install repositories in their respective environments
import os
import yaml
from pathlib import Path

print('📦 Setting up repositories...')
print('='*60)

# ==================== Clone Evo-1 ====================
print('\n[1/3] Cloning Evo-1...')
if not Path('/content/Evo-1/.git').exists():
    !git clone https://github.com/MINT-SJTU/Evo-1.git /content/Evo-1
    print('✅ Cloned Evo-1')
else:
    print('✅ Evo-1 already cloned')

# Install Evo-1 dependencies in server environment
print('\n📦 Installing Evo-1 dependencies in evo1_server...')
evo1_requirements = Path('/content/Evo-1/Evo_1/requirements.txt')
if evo1_requirements.exists():
    !conda run -n evo1_server pip install -q -r /content/Evo-1/Evo_1/requirements.txt
    print('✅ Evo-1 dependencies installed in evo1_server')
else:
    print('⚠️ WARNING: Evo-1 requirements.txt not found')

# ==================== Clone LIBERO ====================
print('\n[2/3] Cloning LIBERO...')
if not Path('/content/LIBERO_evaluation/LIBERO/.git').exists():
    !git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git /content/LIBERO_evaluation/LIBERO
    print('✅ Cloned LIBERO')
else:
    print('✅ LIBERO already cloned')

# Install LIBERO in client environment (Python 3.8.13)
print('\n📦 Installing LIBERO in libero_client...')
libero_requirements = Path('/content/LIBERO_evaluation/LIBERO/requirements.txt')
if libero_requirements.exists():
    !conda run -n libero_client pip install -q -r /content/LIBERO_evaluation/LIBERO/requirements.txt
    !conda run -n libero_client pip install -q -e /content/LIBERO_evaluation/LIBERO
    print('✅ LIBERO installed in libero_client')
else:
    print('⚠️ WARNING: LIBERO requirements.txt not found')

# ==================== Configure LIBERO ====================
print('\n[3/3] Configuring LIBERO...')
os.makedirs(os.path.expanduser('~/.libero'), exist_ok=True)
libero_cfg = os.path.expanduser('~/.libero/config.yaml')

if not os.path.exists(libero_cfg):
    cfg = {
        'benchmark_root': '/content/LIBERO_evaluation/LIBERO/libero/libero',
        'bddl_files': '/content/LIBERO_evaluation/LIBERO/libero/libero/bddl_files',
        'init_states': '/content/LIBERO_evaluation/LIBERO/libero/libero/init_files',
        'datasets': '/content/LIBERO_evaluation/LIBERO/datasets',
        'assets': '/content/LIBERO_evaluation/LIBERO/libero/libero/assets'
    }
    with open(libero_cfg, 'w') as f:
        yaml.dump(cfg, f)
    print('✅ LIBERO config created at ~/.libero/config.yaml')
else:
    print('✅ LIBERO config already exists')

# ==================== Install MetaWorld ====================
print('\n📦 Installing MetaWorld in metaworld_client...')
!conda run -n metaworld_client pip install -q metaworld
!conda run -n metaworld_client pip install -q opencv-python
print('✅ MetaWorld and dependencies installed')

# ==================== Verification ====================
print('\n' + '='*60)
print('🔍 Verifying installations...')
print('='*60)

verification_commands = [
    ('evo1_server', 'python -c "import torch; print(f\'PyTorch: {torch.__version__}\')"'),
    ('libero_client', 'python -c "import libero; print(\'LIBERO imported successfully\')"'),
    ('metaworld_client', 'python -c "import metaworld; print(\'MetaWorld imported successfully\')"'),
]

for env_name, cmd in verification_commands:
    print(f'\n{env_name}:')
    !conda run -n {env_name} {cmd}

print('\n' + '='*60)
print('✅ All repositories installed and configured!')
print('='*60)

📦 Setting up repositories...
✅ Cloned Evo-1
📦 Installing Evo-1 package in evo1_server...
ERROR: file:///content/Evo-1/Evo_1 does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.

ERROR conda.cli.main_run:execute(127): `conda run pip install -q -e /content/Evo-1/Evo_1` failed. (See above for error)
✅ Evo-1 installed in evo1_server
✅ Cloned LIBERO
📦 Installing LIBERO in libero_client...
/content/LIBERO_evaluation/LIBERO
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.2/829.2 kB 41.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.5/734.5 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 MB 122.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.5/193.5 MB 45.7 MB/s 

In [9]:
# 4. Download checkpoints
from huggingface_hub import snapshot_download

CHECKPOINTS_DIR = Path('/content/checkpoints/')

CHECKPOINTS = {
    'libero': {'repo': 'MINT-SJTU/Evo1_LIBERO', 'dir': CHECKPOINTS_DIR / 'libero'},
    'metaworld': {'repo': 'MINT-SJTU/Evo1_MetaWorld', 'dir': CHECKPOINTS_DIR / 'metaworld'}
}

print('📥 Downloading checkpoints...')
for name, cfg in CHECKPOINTS.items():
    cfg['dir'].mkdir(parents=True, exist_ok=True)
    model_file = cfg['dir'] / 'mp_rank_00_model_states.pt'

    if model_file.exists() and model_file.stat().st_size > 1_000_000:
        print(f"✅ {name}: {model_file.stat().st_size / 1e9:.2f} GB")
    else:
        print(f"⏳ Downloading {name}...")
        snapshot_download(
            repo_id=cfg['repo'],
            local_dir=str(cfg['dir']),
            local_dir_use_symlinks=False,
        )
        print(f"✅ {name} downloaded")

print('\n✅ Checkpoints ready')

📥 Downloading checkpoints...
⏳ Downloading libero...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/990 [00:00<?, ?B/s]

norm_stats.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

checkpoint.json:   0%|          | 0.00/89.0 [00:00<?, ?B/s]

norm_stats.json:   0%|          | 0.00/581 [00:00<?, ?B/s]

mp_rank_00_model_states.pt:   0%|          | 0.00/1.55G [00:00<?, ?B/s]

✅ metaworld downloaded

✅ Checkpoints ready


In [ ]:
# 5. Create and patch client scripts for 5 parallel trials
import re
import shutil
from pathlib import Path

# Configuration
NUM_TRIALS = 5
LIBERO_BASE_PORT = 9001  # Ports: 9001-9005
METAWORLD_BASE_PORT = 9101  # Ports: 9101-9105

print('📝 Creating client scripts for parallel trials...')
print(f'   Creating {NUM_TRIALS} LIBERO clients and {NUM_TRIALS} MetaWorld clients')
print('='*60)

# ==================== Patch LIBERO Clients ====================
print(f'\n[1/2] Patching {NUM_TRIALS} LIBERO client scripts...')

libero_client_base = '/content/Evo-1/LIBERO_evaluation/libero_client_4tasks.py'

if not Path(libero_client_base).exists():
    print(f'⚠️ WARNING: {libero_client_base} not found! Skipping LIBERO patching.')
else:
    with open(libero_client_base, 'r') as f:
        content = f.read()

    # Apply base patches
    content = content.replace('os.environ["MUJOCO_GL"] = "osmesa"', 'os.environ["MUJOCO_GL"] = "egl"')

    if 'max_size=' not in content:
        content = re.sub(
            r'async with websockets\.connect\(([^,\)]+)\)',
            r'async with websockets.connect(\1, max_size=100*1024*1024)',
            content
        )

    # Save base version
    with open(libero_client_base, 'w') as f:
        f.write(content)

    # Create trial-specific versions
    for i in range(1, NUM_TRIALS + 1):
        libero_client_patched = f'/content/Evo-1/LIBERO_evaluation/libero_client_trial_{i}.py'
        shutil.copy(libero_client_base, libero_client_patched)

        port = LIBERO_BASE_PORT + i - 1

        with open(libero_client_patched, 'r') as f:
            content = f.read()

        # Patch server URL for this trial
        content = re.sub(
            r'SERVER_URL\s*=\s*"[^"]*"',
            f'SERVER_URL = "ws://localhost:{port}"',
            content
        )

        content = re.sub(
            r'async with websockets\.connect\(([^,\)]+)\)',
            r'async with websockets.connect(\1, max_size=100*1024*1024, ping_interval=120, ping_timeout=300)',
            content
        )

        content = re.sub(
            r'ckpt_name\s*=\s*f?"Evo1_libero_all"',
            f'ckpt_name = "Evo1_libero_trial_{i}"',
            content
        )

        with open(libero_client_patched, 'w') as f:
            f.write(content)

        print(f'  ✅ libero_client_trial_{i}.py → port {port}')

# ==================== Patch MetaWorld Clients ====================
print(f'\n[2/2] Patching {NUM_TRIALS} MetaWorld client scripts...')

mw_client_base = '/content/Evo-1/MetaWorld_evaluation/mt50_evo1_client_prompt.py'

if not Path(mw_client_base).exists():
    print(f'⚠️ WARNING: {mw_client_base} not found! Skipping MetaWorld patching.')
else:
    # First, patch the base MetaWorld client
    with open(mw_client_base, 'r') as f:
        content = f.read()

    # Apply base patches
    content = re.sub(
        r'SHOW_WINDOW\s*=\s*True',
        'SHOW_WINDOW = False',
        content
    )

    # Fix ORDER_JSON_PATH to absolute path
    content = re.sub(
        r'ORDER_JSON_PATH\s*=\s*"mt50_order\.json"',
        'ORDER_JSON_PATH = "/content/Evo-1/MetaWorld_evaluation/mt50_order.json"',
        content
    )

    # Fix TASKS_JSONL_PATH to absolute path
    content = re.sub(
        r'TASKS_JSONL_PATH\s*=\s*"tasks\.jsonl"',
        'TASKS_JSONL_PATH = "/content/Evo-1/MetaWorld_evaluation/tasks.jsonl"',
        content
    )

    # Fix websocket max_size
    if 'max_size=' not in content:
        content = re.sub(
            r'async with websockets\.connect\(server_url\)',
            r'async with websockets.connect(server_url, max_size=100_000_000)',
            content
        )

    # Save base version
    with open(mw_client_base, 'w') as f:
        f.write(content)

    # Create trial-specific versions
    for i in range(1, NUM_TRIALS + 1):
        mw_client_patched = f'/content/Evo-1/MetaWorld_evaluation/mt50_evo1_client_trial_{i}.py'
        shutil.copy(mw_client_base, mw_client_patched)

        port = METAWORLD_BASE_PORT + i - 1

        with open(mw_client_patched, 'r') as f:
            content = f.read()

        # Patch server URL for this trial
        content = re.sub(
            r'SERVER_URL\s*=\s*"[^"]*"',
            f'SERVER_URL = "ws://localhost:{port}"',
            content
        )

        # Update log paths to be trial-specific
        content = re.sub(
            r'LOG_DIR\s*=\s*"[^"]*"',
            f'LOG_DIR = "/content/metaworld_logs_trial_{i}"',
            content
        )

        content = re.sub(
            r'VIDEO_SAVE_DIR\s*=\s*"[^"]*"',
            f'VIDEO_SAVE_DIR = "/content/metaworld_videos_trial_{i}"',
            content
        )

        content = re.sub(
            r'INSPECT_DIR\s*=\s*"[^"]*"',
            f'INSPECT_DIR = "/content/metaworld_frames_trial_{i}"',
            content
        )

        with open(mw_client_patched, 'w') as f:
            f.write(content)

        print(f'  ✅ mt50_evo1_client_trial_{i}.py → port {port}')

print('\n' + '='*60)
print(f'✅ All client scripts ready!')
print(f'   LIBERO: {NUM_TRIALS} clients (ports {LIBERO_BASE_PORT}-{LIBERO_BASE_PORT + NUM_TRIALS - 1})')
print(f'   MetaWorld: {NUM_TRIALS} clients (ports {METAWORLD_BASE_PORT}-{METAWORLD_BASE_PORT + NUM_TRIALS - 1})')
print('='*60)

# ==================== Verify Patches ====================
print('\n🔍 Verifying patches...')

# Verify LIBERO
for i in range(1, NUM_TRIALS + 1):
    client_path = Path(f'/content/Evo-1/LIBERO_evaluation/libero_client_trial_{i}.py')
    if client_path.exists():
        with open(client_path, 'r') as f:
            content = f.read()
        expected_port = LIBERO_BASE_PORT + i - 1
        if f'ws://localhost:{expected_port}' in content:
            print(f'  ✅ LIBERO trial {i}: Correct port {expected_port}')
        else:
            print(f'  ❌ LIBERO trial {i}: Port patch failed!')
    else:
        print(f'  ❌ LIBERO trial {i}: File not created!')

# Verify MetaWorld
for i in range(1, NUM_TRIALS + 1):
    client_path = Path(f'/content/Evo-1/MetaWorld_evaluation/mt50_evo1_client_trial_{i}.py')
    if client_path.exists():
        with open(client_path, 'r') as f:
            content = f.read()
        expected_port = METAWORLD_BASE_PORT + i - 1
        if f'ws://localhost:{expected_port}' in content:
            print(f'  ✅ MetaWorld trial {i}: Correct port {expected_port}')
        else:
            print(f'  ❌ MetaWorld trial {i}: Port patch failed!')
    else:
        print(f'  ❌ MetaWorld trial {i}: File not created!')

print('\n✅ Patching verification complete!')

📝 Patching client scripts...
✅ LIBERO client patched
📝 Patching MetaWorld client...
✅ MetaWorld client patched

✅ Client scripts ready
   Note: Using standard Evo-1 servers (no ablation)


In [ ]:
# 6. Run benchmarks - Sequential execution (MetaWorld first, then LIBERO)
import subprocess
import time
import os
import torch
from pathlib import Path

LOGS_DIR = Path('/content/logs')
LOGS_DIR.mkdir(parents=True, exist_ok=True)

# Configuration
NUM_TRIALS = 5
LIBERO_BASE_PORT = 9001  # Ports: 9001-9005
METAWORLD_BASE_PORT = 9101  # Ports: 9101-9105

def cleanup_ports(ports):
    """Kill processes on specified ports"""
    for port in ports:
        subprocess.run(f"lsof -ti:{port} | xargs -r kill -9 2>/dev/null", shell=True)
    subprocess.run("pkill -f 'Evo1.*server' 2>/dev/null", shell=True)

def check_port(port, timeout=5):
    try:
        import websocket
        ws = websocket.create_connection(f"ws://localhost:{port}", timeout=timeout)
        ws.close()
        return True
    except Exception:
        return False

def wait_for_server(port, name, max_wait=180):
    print(f"  Waiting for {name} on port {port}...", end="", flush=True)
    start = time.time()
    while time.time() - start < max_wait:
        if check_port(port):
            print(f" ✅ ready ({int(time.time()-start)}s)")
            return True
        time.sleep(2)
        print(".", end="", flush=True)
    print(f" ❌ timeout")
    return False

def check_gpu_memory():
    """Return available GPU memory in GB"""
    result = subprocess.run(['nvidia-smi', '--query-gpu=memory.free', '--format=csv,noheader,nounits'],
                          capture_output=True, text=True)
    return int(result.stdout.strip()) / 1024

def monitor_processes(processes, benchmark_name):
    """Monitor processes and return when all complete"""
    try:
        while True:
            time.sleep(60)
            print('\n' + '='*60)
            print(f'--- {benchmark_name} Status ({time.strftime("%H:%M:%S")}) ---')
            print('='*60)

            all_done = True
            running_count = 0
            failed_count = 0
            done_count = 0

            # Group by type
            servers = [p for p in processes if p[0].startswith('server_')]
            clients = [p for p in processes if p[0].startswith('client_')]

            for group_name, group in [("Servers", servers), ("Clients", clients)]:
                print(f"\n{group_name}:")
                for name, p, _ in group:
                    if p.poll() is None:
                        print(f"  🟢 {name}: Running")
                        all_done = False
                        running_count += 1
                    else:
                        if p.returncode == 0:
                            print(f"  ✅ {name}: Done")
                            done_count += 1
                        else:
                            print(f"  ❌ {name}: Failed (code={p.returncode})")
                            failed_count += 1

            print(f"\n📊 Summary: {running_count} running, {done_count} done, {failed_count} failed")

            if all_done:
                print('\n' + '='*60)
                print(f'✅ {benchmark_name} completed!')
                print('='*60)
                break

    except KeyboardInterrupt:
        print(f'\n🛑 Stopping {benchmark_name}...')
        for name, p, log in processes:
            if p.poll() is None:
                print(f"  Terminating {name}...")
                p.terminate()
            log.close()
        raise

# ============================================================================
# PHASE 1: MetaWorld (5 parallel trials)
# ============================================================================

print('\n' + '='*80)
print('🎯 PHASE 1: MetaWorld Baseline Study (5 Parallel Trials)')
print('='*80)

# Cleanup
print('\n🧹 Cleaning up existing processes...')
metaworld_ports = list(range(METAWORLD_BASE_PORT, METAWORLD_BASE_PORT + NUM_TRIALS))
cleanup_ports(metaworld_ports)
time.sleep(2)

# Check GPU
free_mem = check_gpu_memory()
print(f'💾 GPU free memory: {free_mem:.1f}GB')

processes = []
env = {**os.environ, "PYTHONUNBUFFERED": "1"}

# Start MetaWorld Servers
print(f'\n📡 Starting {NUM_TRIALS} MetaWorld baseline servers...')
for i in range(1, NUM_TRIALS + 1):
    port = METAWORLD_BASE_PORT + i - 1
    trial_name = f"MetaWorld_Baseline_Trial_{i}"

    free_mem = check_gpu_memory()
    print(f"  [Trial {i}] GPU free: {free_mem:.1f}GB", end=" ")

    cmd = f"conda run --no-capture-output -n evo1_server python -u /content/Evo-1/Evo_1/scripts/Evo1_server.py --port {port} --checkpoint {CHECKPOINTS_DIR}/metaworld --name {trial_name}"

    log = open(f"{LOGS_DIR}/server_metaworld_baseline_trial_{i}.log", "w")
    p = subprocess.Popen(cmd, shell=True, stdout=log, stderr=log, env=env)
    processes.append((f"server_metaworld_{i}", p, log))

    if not wait_for_server(port, f"MW-{i}"):
        raise RuntimeError(f"MetaWorld server {i} failed to start on port {port}")

print(f'\n✅ All {NUM_TRIALS} MetaWorld servers ready!')

# Start MetaWorld Clients
print(f'\n🔬 Starting {NUM_TRIALS} MetaWorld clients...')
for i in range(1, NUM_TRIALS + 1):
    port = METAWORLD_BASE_PORT + i - 1
    patched_script = f'/content/Evo-1/MetaWorld_evaluation/mt50_evo1_client_trial_{i}.py'

    cmd = f"conda run --no-capture-output -n metaworld_client python -u {patched_script}"
    log = open(f"{LOGS_DIR}/client_metaworld_baseline_trial_{i}.log", "w")
    p = subprocess.Popen(cmd, shell=True, stdout=log, stderr=log, stdin=subprocess.DEVNULL, env=env)
    processes.append((f"client_metaworld_{i}", p, log))
    print(f'  ✅ MetaWorld client {i} started (port {port})')

print('\n' + '='*60)
print(f'✅ MetaWorld benchmarks running ({NUM_TRIALS} trials)')
print('='*60)
print(f'Logs: {LOGS_DIR}/server_metaworld_baseline_trial_*.log')
print(f'      {LOGS_DIR}/client_metaworld_baseline_trial_*.log')
print('\n⏳ Monitoring MetaWorld trials...')

# Monitor MetaWorld
monitor_processes(processes, "MetaWorld")

# Cleanup MetaWorld
print('\n🧹 Cleaning up MetaWorld servers...')
for name, p, log in processes:
    if p.poll() is None:
        p.terminate()
    log.close()

cleanup_ports(metaworld_ports)
time.sleep(3)

# Free GPU memory
print('🔄 Freeing GPU memory...')
torch.cuda.empty_cache()
time.sleep(2)

free_mem = check_gpu_memory()
print(f'💾 GPU free memory after cleanup: {free_mem:.1f}GB')

🧹 Cleaning up...
🚀 Starting baseline Evo-1 servers (evo1_server environment)...
✅ Patched server for LIBERO:
   - Checkpoint: /content/checkpoints/libero
   - Port: 9001
  Waiting for LIBERO on port 9001...... ✅ ready (15s)
✅ Patched server for MetaWorld:
   - Checkpoint: /content/checkpoints/metaworld
   - Port: 9002
  Waiting for MetaWorld on port 9002...... ✅ ready (15s)

✅ All servers ready!

🔬 Starting benchmark clients (separate environments)...
  ✅ LIBERO client started (Python 3.8.13)
  ✅ MetaWorld client started (Python 3.10)

✅ Baseline benchmarks running (standard Evo-1 with full state)!
Logs: /content/logs/

⏳ Monitoring... (This will take ~30-60 min)

--- Status ---
  server_libero: Running
  server_metaworld: Running
  client_libero: Running
  client_metaworld: Running

--- Status ---
  server_libero: Running
  server_metaworld: Running
  client_libero: Running
  client_metaworld: Running

--- Status ---
  server_libero: Running
  server_metaworld: Running
  client_libero

In [ ]:
# 7. LIBERO Baseline (5 parallel trials)
import subprocess
import time
import os
import torch
from pathlib import Path

# ============================================================================
# PHASE 2: LIBERO (5 parallel trials)
# ============================================================================

print('\n' + '='*80)
print('🎯 PHASE 2: LIBERO Baseline Study (5 Parallel Trials)')
print('='*80)

# Cleanup
print('\n🧹 Cleaning up existing processes...')
libero_ports = list(range(LIBERO_BASE_PORT, LIBERO_BASE_PORT + NUM_TRIALS))
cleanup_ports(libero_ports)
time.sleep(2)

# Check GPU
free_mem = check_gpu_memory()
print(f'💾 GPU free memory: {free_mem:.1f}GB')

processes = []

# Start LIBERO Servers
print(f'\n📡 Starting {NUM_TRIALS} LIBERO baseline servers...')
for i in range(1, NUM_TRIALS + 1):
    port = LIBERO_BASE_PORT + i - 1
    trial_name = f"LIBERO_Baseline_Trial_{i}"

    free_mem = check_gpu_memory()
    print(f"  [Trial {i}] GPU free: {free_mem:.1f}GB", end=" ")

    cmd = f"conda run --no-capture-output -n evo1_server python -u /content/Evo-1/Evo_1/scripts/Evo1_server.py --port {port} --checkpoint {CHECKPOINTS_DIR}/libero --name {trial_name}"

    log = open(f"{LOGS_DIR}/server_libero_baseline_trial_{i}.log", "w")
    p = subprocess.Popen(cmd, shell=True, stdout=log, stderr=log, env=env)
    processes.append((f"server_libero_{i}", p, log))

    if not wait_for_server(port, f"LIBERO-{i}"):
        raise RuntimeError(f"LIBERO server {i} failed to start on port {port}")

print(f'\n✅ All {NUM_TRIALS} LIBERO servers ready!')

# Start LIBERO Clients
print(f'\n🔬 Starting {NUM_TRIALS} LIBERO clients...')
for i in range(1, NUM_TRIALS + 1):
    port = LIBERO_BASE_PORT + i - 1
    patched_script = f'/content/Evo-1/LIBERO_evaluation/libero_client_trial_{i}.py'

    cmd = f"conda run --no-capture-output -n libero_client python -u {patched_script}"
    log = open(f"{LOGS_DIR}/client_libero_baseline_trial_{i}.log", "w")
    client_env = {**env, "PYTHONPATH": "/content/LIBERO_evaluation/LIBERO"}
    p = subprocess.Popen(cmd, shell=True, stdout=log, stderr=log, stdin=subprocess.DEVNULL, env=client_env)
    processes.append((f"client_libero_{i}", p, log))
    print(f'  ✅ LIBERO client {i} started (port {port})')

print('\n' + '='*60)
print(f'✅ LIBERO benchmarks running ({NUM_TRIALS} trials)')
print('='*60)
print(f'Logs: {LOGS_DIR}/server_libero_baseline_trial_*.log')
print(f'      {LOGS_DIR}/client_libero_baseline_trial_*.log')
print('\n⏳ Monitoring LIBERO trials...')

# Monitor LIBERO
monitor_processes(processes, "LIBERO")

# Final cleanup
print('\n🧹 Final cleanup...')
for name, p, log in processes:
    if p.poll() is None:
        p.terminate()
    log.close()

cleanup_ports(libero_ports)

# ============================================================================
# Summary
# ============================================================================

print('\n' + '='*80)
print('📊 BASELINE STUDY COMPLETE')
print('='*80)
print(f'\n✅ Phase 1: MetaWorld ({NUM_TRIALS} trials) - COMPLETE')
print(f'✅ Phase 2: LIBERO ({NUM_TRIALS} trials) - COMPLETE')
print(f'\n📁 All logs saved to: {LOGS_DIR}/')
print(f'\nMetaWorld:')
print(f'  - Server logs: server_metaworld_baseline_trial_{{1-{NUM_TRIALS}}}.log')
print(f'  - Client logs: client_metaworld_baseline_trial_{{1-{NUM_TRIALS}}}.log')
print(f'  - Videos: /content/metaworld_videos_trial_{{1-{NUM_TRIALS}}}/')
print(f'\nLIBERO:')
print(f'  - Server logs: server_libero_baseline_trial_{{1-{NUM_TRIALS}}}.log')
print(f'  - Client logs: client_libero_baseline_trial_{{1-{NUM_TRIALS}}}.log')
print('\n💡 Next steps:')
print('  1. Review logs for any errors')
print('  2. Archive results to Drive')
print('  3. Compare with ablation results')
print('='*80)

In [ ]:
# 8. View MetaWorld results
print('📊 MetaWorld Baseline Results\n')
print('='*60)

for i in range(1, NUM_TRIALS + 1):
    print(f'\n--- MetaWorld Trial {i} ---')
    !tail -30 {LOGS_DIR}/client_metaworld_baseline_trial_{i}.log 2>/dev/null || echo 'No results yet'
    print('='*60)

Streaming output truncated to the last 5000 lines.
[-0.10532820224761963, -0.0032399892807006836, 0.10638368129730225, -0.041933536529541016, -0.0071623921394348145, -0.01710018515586853, 0.001992166042327881]
gripper action 1
[Step 4] reward=0.00, done=False
[-0.11572730541229248, -0.013262569904327393, 0.13139140605926514, -0.0397317111492157, -0.010090351104736328, -0.014534562826156616, -0.0005685091018676758]
gripper action 1
[Step 4] reward=0.00, done=False
[-0.11306250095367432, -0.010344326496124268, 0.19694817066192627, -0.0382838100194931, -0.01410830020904541, -0.011428296566009521, -6.121397018432617e-05]
gripper action 1
[Step 4] reward=0.00, done=False
[-0.17157244682312012, -0.015104413032531738, 0.2557905912399292, -0.0357828289270401, -0.014410614967346191, -0.012017369270324707, 0.0008423924446105957]
gripper action 1
[Step 4] reward=0.00, done=False
[-0.19292861223220825, 0.02152252197265625, 0.31979799270629883, -0.029717475175857544, -0.015537798404693604, -0.01621

In [ ]:
# 9. View LIBERO results
print('📊 LIBERO Baseline Results\n')
print('='*60)

for i in range(1, NUM_TRIALS + 1):
    print(f'\n--- LIBERO Trial {i} ---')
    !tail -30 {LOGS_DIR}/client_libero_baseline_trial_{i}.log 2>/dev/null || echo 'No results yet'
    print('='*60)

In [ ]:
# 10. Save MetaWorld results to Drive
import os
import shutil
import zipfile

src_dir = "/content"
dst_dir = "/content/drive/MyDrive/evo1_baseline/Results/MetaworldBaselinePass0"
zip_path = "/content/metaworld_baseline_results.zip"

os.makedirs(dst_dir, exist_ok=True)

prefixes = ("metaworld_logs", "metaworld_video")

# Create zip file
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for name in os.listdir(src_dir):
        src_path = os.path.join(src_dir, name)
        
        if name.startswith(prefixes):
            if os.path.isdir(src_path):
                # Add directory to zip
                for root, dirs, files in os.walk(src_path):
                    for file in files:
                        file_path = os.path.join(root, file)
                        arcname = os.path.relpath(file_path, src_dir)
                        zipf.write(file_path, arcname)
            else:
                # Add file to zip
                zipf.write(src_path, name)

# Copy zip file to Drive
shutil.copy2(zip_path, os.path.join(dst_dir, "metaworld_baseline_results.zip"))

print(f'✅ MetaWorld results saved to: {dst_dir}')
print(f'   Archive: metaworld_baseline_results.zip')

In [ ]:
# 11. Save LIBERO results to Drive
import os
import shutil

src_dir = "/content"
dst_dir = "/content/drive/MyDrive/evo1_baseline/Results/LIBEROBaselinePass0"

os.makedirs(dst_dir, exist_ok=True)

prefixes = ("logs", "video")

for name in os.listdir(src_dir):
    src_path = os.path.join(src_dir, name)
    dst_path = os.path.join(dst_dir, name)

    if name.startswith(prefixes):
        if os.path.isdir(src_path):
            if os.path.exists(dst_path):
                shutil.rmtree(dst_path)
            shutil.copytree(src_path, dst_path)
        else:
            shutil.copy2(src_path, dst_path)

print(f'✅ LIBERO results saved to: {dst_dir}')